In [ ]:
from sqlalchemy import create_engine
import urllib.parse
import os

server = os.getenv("OLIST_SQL_SERVER", "YOUR_SERVER_NAME")
database = "Olist_Ecommerce"
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

params = urllib.parse.quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

print("SQL Server connection engine configured successfully!")

In [ ]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd()
folder_path = project_path / "Raw_Data"

files = folder_path.glob("*.csv")
data = {}

for file_path in files:
    data[file_path.stem] = pd.read_csv(file_path)

for name, df in data.items():
    print(name, df.shape)

In [ ]:
clean_folder = project_path / "Clean_Dataset"

clean_folder.mkdir(parents=True, exist_ok=True)

print("Clean folder ready!")

In [ ]:
orders_df = data["olist_orders_dataset"]

customers_df = data["olist_customers_dataset"]

products_df = data["olist_products_dataset"]

order_items_df = data["olist_order_items_dataset"]

payments_df = data["olist_order_payments_dataset"]

reviews_df = data["olist_order_reviews_dataset"]

sellers_df = data["olist_sellers_dataset"]

geo_df = data["olist_geolocation_dataset"]

closed_deals_df = data["olist_closed_deals_dataset"]

category_translation_df = data["product_category_name_translation"]

In [ ]:
orders_clean = orders_df.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for dt in date_columns:
    orders_clean[dt] = pd.to_datetime(orders_clean[dt])

orders_clean.to_csv(clean_folder / "orders_clean.csv", index=False)


customers_clean = customers_df.copy()
customers_clean.to_csv(clean_folder / "customers_clean.csv", index=False)


products_clean = products_df.copy()
products_clean.to_csv(clean_folder / "products_clean.csv", index=False)


order_items_clean = order_items_df.copy()
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"]
)
order_items_clean.to_csv(clean_folder / "order_items_clean.csv", index=False)


payments_clean = payments_df.copy()
payments_clean.to_csv(clean_folder / "payments_clean.csv", index=False)


reviews_clean = reviews_df.copy()

date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for dt in date_columns:
    reviews_clean[dt] = pd.to_datetime(reviews_clean[dt])

reviews_clean.to_csv(clean_folder / "reviews_clean.csv", index=False)


sellers_clean = sellers_df.copy()
sellers_clean.to_csv(clean_folder / "sellers_clean.csv", index=False)


geo_clean = geo_df.copy()
geo_clean.to_csv(clean_folder / "geo_clean.csv", index=False)


category_translation_clean = category_translation_df.copy()
category_translation_clean.to_csv(
    clean_folder / "category_translation_clean.csv",
    index=False
)

print("All 9 cleaned datasets exported successfully!")

In [ ]:
for file_path in sorted(clean_folder.iterdir()):
    if file_path.is_file():
        print(file_path.name)

In [ ]:
tables = {
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "products": products_clean,
    "reviews": reviews_clean,
    "sellers": sellers_clean,
    "geolocation": geo_clean,
    "category_translation": category_translation_clean,
    "customers": customers_clean
}

print("WARNING: This operation replaces existing SQL tables.")
confirm_upload = input("Type YES to continue: ")

if confirm_upload == "YES":
    for table_name, df in tables.items():
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",
            index=False,
            chunksize=5000
        )
        print(f"{table_name} uploaded successfully!")
else:
    print("SQL upload cancelled.")

In [ ]:
from sqlalchemy import text

for table_name, df in {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "products": products_clean,
    "reviews": reviews_clean,
    "sellers": sellers_clean,
    "geolocation": geo_clean,
    "category_translation": category_translation_clean
}.items():

    with engine.connect() as conn:
        sql_count = conn.execute(
            text(f"SELECT COUNT(*) FROM dbo.[{table_name}]")
        ).scalar_one()

    status = "MATCH" if len(df) == sql_count else "MISMATCH"

    print(
        f"{table_name}: "
        f"Python = {len(df):,} | "
        f"SQL = {sql_count:,} | "
        f"{status}"
    )